In [ ]:
INPUT_TEXT = "What is a Transformers in Deep Learning?"
MODEL_NAME = "Qwen/Qwen3-1.7B"

In [ ]:
import torch
import setuptools  # loads distutils shim for Python 3.13+
from torchviz import make_dot
from transformers import AutoModelForCausalLM, AutoTokenizer

In [16]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=None,
    trust_remote_code=True
)
qwen1_7B = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=None,
    torch_dtype=torch.float16,
    device_map="auto"
)


qwen1_7B.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [ ]:
# Isolate Blocks
model_block = qwen1_7B.model
lm_head = qwen1_7B.lm_head

embed_block = model_block.embed_tokens
norm_block = model_block.norm
rotary_emb_block = model_block.rotary_emb

transformer_block = model_block.layers

attn_block = transformer_block[0].self_attn
ffn_block = transformer_block[0].mlp
input_layernorm = transformer_block[0].input_layernorm
post_attn_layernorm = transformer_block[0].post_attention_layernorm


Qwen3MLP(
  (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
  (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
  (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
  (act_fn): SiLUActivation()
)

In [14]:
prompt = "Explain what a Transformer architecture is in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(qwen1_7B.device)

with torch.no_grad():
    output_ids = qwen1_7B.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )[0]

generated = tokenizer.decode(output_ids[inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(generated)


 The Transformer architecture is a type of neural network that uses self-attention mechanisms to process sequences of data. It was introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. The key innovation is that instead of using recurrent neural networks (RNNs) or convolutional neural networks (CNNs), the Transformer uses self-attention to handle long-range dependencies in the input. This allows the model to capture relationships between different parts of the input sequence, even if they are far apart. The self-attention mechanism works by creating a query, key, and value matrix for each position in the sequence. The model then computes the attention scores for each pair of positions, which are used to weight the values of the input sequence. The weighted sum of these values is used as the output for each position in the sequence. The Transformer architecture is known for its efficiency and effectiveness in various tasks, including natural language processing,

In [ ]:
# Trace just ONE decoder layer (not all 28) to see the architecture without repetition
single_layer = qwen1_7B.model.layers[0]
dummy_hidden = torch.randn(1, 16, 2048, dtype=torch.float16).to(qwen1_7B.device)
position_ids = torch.arange(16, dtype=torch.long).unsqueeze(0).to(qwen1_7B.device)

# The layer needs RoPE embeddings (cos, sin) from the parent model's rotary_emb
position_embeddings = qwen1_7B.model.rotary_emb(dummy_hidden, position_ids)

layer_out = single_layer(dummy_hidden, position_ids=position_ids, position_embeddings=position_embeddings)
make_dot(layer_out[0], params=dict(single_layer.named_parameters()))